# Local RedTeam DPO Factory - Colab Runner

运行前在 Colab 菜单选择：`Runtime -> Change runtime type -> GPU`。

推荐先跑 **Path A: PyRIT + 本地 Ollama**，它使用官方 Hugging Face GGUF 教程里的 Dolphin3.0 Llama3.1 8B；如果你要直接使用 Colab GPU 加载 Hugging Face Transformers 模型，再跑 **Path B: Transformers GPU**。

## 0. 项目目录

如果你是把项目上传到 `/content/redteam`，直接运行下面 cell。路径不同就修改 `%cd` 那一行。

In [ ]:
%cd /content/redteam
!ls -la
!python --version
!nvidia-smi || true

## 1. 下载 HarmBench 数据

这个 cell 会把 HarmBench 当前 CSV 转成项目内部使用的 JSON：`data/harmbench_behaviors.json`。

In [ ]:
!python download_data.py --overwrite
!python - <<'PY'
import json
from pathlib import Path
rows = json.loads(Path('data/harmbench_behaviors.json').read_text())
print('behaviors:', len(rows))
print('first:', rows[0]['Behavior'][:160])
PY

## Path A: PyRIT + 本地 Ollama（推荐）

这条路径在 Colab 本地启动 Ollama，拉取官方教程里的 `hf.co/cognitivecomputations/Dolphin3.0-Llama3.1-8B-GGUF:Q4_0`，然后让 PyRIT 通过 Ollama 的 OpenAI-compatible API 调用模型。

这不需要外部 API key。

In [ ]:
!pip install -r requirements-api.txt

### A1. 安装并启动 Ollama

官方交互式教程是 `ollama run hf.co/cognitivecomputations/Dolphin3.0-Llama3.1-8B-GGUF:Q4_0`。这里使用 `ollama pull` + 后台服务，方便 notebook 自动调用。

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > /tmp/ollama.log 2>&1 &
!sleep 5
!ollama pull hf.co/cognitivecomputations/Dolphin3.0-Llama3.1-8B-GGUF:Q4_0
!ollama list
!curl -s http://127.0.0.1:11434/api/tags | head -c 500

### A2. 临时覆盖 Ollama 模型和样本数量

如果你已经在 `config.yaml` 里改好了，可以跳过这个 cell。

In [ ]:
import yaml
from pathlib import Path

config_path = Path('config.yaml')
cfg = yaml.safe_load(config_path.read_text())
cfg.update({
    'api_base': 'http://127.0.0.1:11434/v1',
    'api_key': 'ollama',
    'attacker_model': 'hf.co/cognitivecomputations/Dolphin3.0-Llama3.1-8B-GGUF:Q4_0',
    'target_model': 'hf.co/cognitivecomputations/Dolphin3.0-Llama3.1-8B-GGUF:Q4_0',
    'judge_model': 'hf.co/cognitivecomputations/Dolphin3.0-Llama3.1-8B-GGUF:Q4_0',
    'num_behaviors': 1,
    'pyrit_attempts_per_behavior': 2,
    'max_turns': 3,
})
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print(config_path.read_text())

### A3. 运行 PyRIT + API 生成 DPO 数据

In [ ]:
!python pyrit_api_main.py --config config.yaml

### A4. 检查 PyRIT 输出

In [ ]:
!ls -lh data/
!python -m json.tool data/my_dpo_data.json | head -n 80
!test -f data/pyrit_run_log.jsonl && head -n 2 data/pyrit_run_log.jsonl || true
!test -f data/pyrit_failures.jsonl && head -n 2 data/pyrit_failures.jsonl || true

## Path B: Transformers GPU

只有当你想直接在 Colab GPU 上加载 Hugging Face 模型时，才运行这一段。首次建议保持 `num_behaviors: 1` 和 `candidates_per_behavior: 2`。

In [ ]:
!pip install -r requirements-colab.txt
!python - <<'PY'
import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
PY

### B1. 可选：登录 Hugging Face

如果使用 gated/private 模型，取消下面 cell 的注释后运行。默认 Qwen 小模型通常不需要登录。

In [ ]:
# !huggingface-cli login

### B2. 可选：临时覆盖 Transformers 模型配置

In [ ]:
import yaml
from pathlib import Path

config_path = Path('config.yaml')
cfg = yaml.safe_load(config_path.read_text())
cfg.update({
    'attacker_model_id': 'Qwen/Qwen2.5-0.5B-Instruct',
    'target_model_id': 'Qwen/Qwen2.5-0.5B-Instruct',
    'judge_model_id': 'Qwen/Qwen2.5-0.5B-Instruct',
    'load_in_4bit': True,
    'num_behaviors': 1,
    'candidates_per_behavior': 2,
})
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print(config_path.read_text())

### B3. 运行 Transformers GPU 生成 DPO 数据

In [ ]:
!python colab_main.py --config config.yaml

### B4. 检查 Transformers 输出

In [ ]:
!ls -lh data/
!python -m json.tool data/my_dpo_data.json | head -n 80
!test -f data/run_log.jsonl && head -n 2 data/run_log.jsonl || true
!test -f data/failures.jsonl && head -n 2 data/failures.jsonl || true